In [0]:
%sql
-- Databricks / Spark SQL

WITH fct_filtered AS (
  -- Factは最初に期間で絞る（I/O削減が最重要）
  SELECT
    fct.time_period_end_date,
    fct.prod_key,
    fct.gross_transact_amt,
    fct.net_transact_amt
  FROM hive_metastore.japan_linkedexcel.shipment_fct_vw AS fct
  WHERE 1 = 1
    -- TODO: 期間条件は必須で入れてください（例）
    -- AND fct.time_period_end_date >= TIMESTAMP('2025-01-01 00:00:00')
    -- AND fct.time_period_end_date <  TIMESTAMP('2025-02-01 00:00:00')
),

calender_mapped AS (
  -- ご要望に合わせて calender 表記で記載（テーブル実在は要確認）
  SELECT
    cal.day_date,
    cal.fy_year_name,
    cal.fy_month_name,
    cal.fy_quarter_name
  FROM hive_metastore.japanlinkedexcel.calender_vw AS cal
  WHERE cal.calendar_type = 'Standard Calender'
),

product_mapped AS (
  -- ext優先：Categoryは prod_dim_ext_vw の jp_category_name を使用
  SELECT /*+ BROADCAST(pe) */
    pe.prod_key,
    pe.jp_category_name
  FROM hive_metastore.japanlinkedexcel.prod_dim_ext_vw AS pe
)

SELECT
  cal.fy_year_name    AS fy,
  cal.fy_month_name   AS period,
  cal.fy_quarter_name AS quarter,

  -- JOIN KEY制約により、shipment -> customer(FN) のJOINができないため現状は出せません
  CAST(NULL AS STRING) AS fn_code,
  CAST(NULL AS STRING) AS fn_name,

  pe.jp_category_name AS category,

  SUM(COALESCE(fct.gross_transact_amt, 0)) AS ships_in_giv,
  SUM(COALESCE(fct.net_transact_amt,   0)) AS ships_in_niv
FROM fct_filtered AS fct
JOIN calender_mapped AS cal
  ON fct.time_period_end_date = cal.day_date   -- ※JOIN KEY側は本来 calendar_dim_vw.day_date 定義（名称差分は要確認）
JOIN product_mapped AS pe
  ON fct.prod_key = pe.prod_key
GROUP BY
  cal.fy_year_name,
  cal.fy_month_name,
  cal.fy_quarter_name,
  pe.jp_category_name
ORDER BY
  fy, period, quarter, category;

In [0]:
%sql

SELECT
  Cal.`PG Year (Real)`,
  Cal.fy_half_year_name,
  gross_transact_amt,
  net_transact_amt,
  Cal.`Day Date (Real)`
FROM 
  hive_metastore.japan_linkedexcel.calender_dim_vw AS Cal
LEFT OUTER JOIN
  hive_metastore.japan_linkedexcel.shipment_fct_vw AS DS1
ON
  Cal.`Day Date (Real)` = DS1.day_date
WHERE
  Cal.`PG Year (Real)` = "FY2526"
LIMIT 10

In [0]:
%sql
-- DSCからTarget取得 with Org
-- FY2425 2H + FN2置き換え（RTはcustomer_japan_dim_vw、WSはwarehouse_dim_vw）

WITH
/* === FY2425 2H の日付集合 === */
cal_fy_2h AS (
  SELECT
    c.day_date,
    c.fy_quarter_name AS pg_quarter
  FROM hive_metastore.japan_gold.calender_dim_vw AS c
  WHERE c.fy_year_name = 'FY2425'
    AND c.fy_half_year_name = '2H'
),

/* === パーティションプルーニング用にstart/endも作る === */
cal_bounds AS (
  SELECT
    MIN(day_date) AS start_ts,
    (MAX(day_date) + INTERVAL 1 DAY) AS end_ts_excl
  FROM cal_fy_2h
),

/* === Org2-5 === */
org AS (
  SELECT
    o.org5_key,
    o.org2_name AS org2,
    o.org3_name AS org3,
    o.org4_name AS org4,
    o.org5_name AS org5
  FROM hive_metastore.japan_gold.sl_organization_dim_vw AS o
  WHERE o.org5_key IS NOT NULL
),

/* === GIV dimension（customer_org_mapping_key起点のマスタ） === */
giv_dim_base AS (
  SELECT
    gd.customer_org_mapping_key,
    gd.cust_key,
    gd.category_name AS category,
    gd.Org_Code      AS org_code,
    gd.org5_key,

    -- giv_dimension_dim_vwには「最新FN」が入っているが、今回はFN2を返したいので原則使わない
    -- ただしFN2がNULLのケースの保険として保持しておく
    gd.FN_Code       AS fn_code_current,
    gd.FN_Name       AS fn_name_current
  FROM hive_metastore.japan_gold.giv_dimension_dim_vw AS gd
),

/* === RT側の将来FN（FN2） === */
rt_cust AS (
  SELECT
    c.cust_key,
    c.jp_cust_parent_group_name AS parent_group_rt,  -- RT/WS判定の材料（RT側）
    c.jp_local_fn_code_2        AS fn2_code_rt,
    c.jp_local_fn_name_2        AS fn2_name_rt
  FROM hive_metastore.japan_gold.customer_japan_dim_vw AS c
),

/* === WS側の将来FN（FN2） === */
ws_cust AS (
  SELECT
    w.cust_key,
    w.jp_cust_parent_group_name AS parent_group_ws,  -- RT/WS判定の材料（WS側）
    w.jp_local_fn_code_2        AS fn2_code_ws,
    w.jp_local_fn_name_2        AS fn2_name_ws
  FROM hive_metastore.japan_gold.warehouse_dim_vw AS w
),

/* === cust_keyごとに「DC/WS」とFN2を確定する（RT/WSで参照先を切替） === */
cust_fn2 AS (
  SELECT
    gdb.cust_key,

    /* 判定優先度：
       - まずWS側マスタにparent_groupがあればそれを優先（WSをより確実に拾う想定）
       - なければRT側を採用
    */
    COALESCE(wc.parent_group_ws, rc.parent_group_rt) AS dc_ws,

    /* RT/WSの値が想定どおり 'RT' / 'WS' の場合の分岐
       ※もし値が異なるならこのCASEを調整してください
    */
    CASE
      WHEN COALESCE(wc.parent_group_ws, rc.parent_group_rt) = 'WS' THEN wc.fn2_code_ws
      ELSE rc.fn2_code_rt
    END AS fn_code_2,

    CASE
      WHEN COALESCE(wc.parent_group_ws, rc.parent_group_rt) = 'WS' THEN wc.fn2_name_ws
      ELSE rc.fn2_name_rt
    END AS fn_name_2

  FROM (SELECT DISTINCT cust_key FROM giv_dim_base) AS gdb
  LEFT JOIN ws_cust AS wc
    ON gdb.cust_key = wc.cust_key
  LEFT JOIN rt_cust AS rc
    ON gdb.cust_key = rc.cust_key
),

/* === Facts（日次で先に集計して行爆発を防ぐ） === */
ship_day AS (
  SELECT
    sh.customer_org_mapping_key,
    sh.day_date,
    SUM(sh.gross_transact_amt) AS shipment_giv_value
  FROM hive_metastore.japan_gold.shipments_fct_vw AS sh
  CROSS JOIN cal_bounds AS b
  WHERE sh.day_date >= b.start_ts
    AND sh.day_date <  b.end_ts_excl
  GROUP BY sh.customer_org_mapping_key, sh.day_date
),

indirect_day AS (
  SELECT
    isd.customer_org_mapping_key,
    isd.day_date,
    SUM(isd.gross_transact_amt) AS indirect_giv_shipments
  FROM hive_metastore.japan_linkedexcel.indirect_ship_day_fct_vw AS isd
  CROSS JOIN cal_bounds AS b
  WHERE isd.day_date >= b.start_ts
    AND isd.day_date <  b.end_ts_excl
  GROUP BY isd.customer_org_mapping_key, isd.day_date
),

garp_day AS (
  SELECT
    g.customer_org_mapping_key,
    g.day_date,
    SUM(g.garp_value) AS garp_value
  FROM hive_metastore.japan_gold.giv_garp_dim_vw AS g
  CROSS JOIN cal_bounds AS b
  WHERE g.day_date >= b.start_ts
    AND g.day_date <  b.end_ts_excl
  GROUP BY g.customer_org_mapping_key, g.day_date
),

outbound_day AS (
  SELECT
    go.customer_org_mapping_key,
    go.day_date,
    SUM(go.giv_outbound) AS giv_outbound
  FROM hive_metastore.japan_gold.giv_outbound_vw AS go
  CROSS JOIN cal_bounds AS b
  WHERE go.day_date >= b.start_ts
    AND go.day_date <  b.end_ts_excl
  GROUP BY go.customer_org_mapping_key, go.day_date
),

/* === 日付×mappingキーの母集合（どれかのfactに存在する日を拾う） === */
all_day_keys AS (
  SELECT customer_org_mapping_key, day_date FROM ship_day
  UNION
  SELECT customer_org_mapping_key, day_date FROM indirect_day
  UNION
  SELECT customer_org_mapping_key, day_date FROM garp_day
  UNION
  SELECT customer_org_mapping_key, day_date FROM outbound_day
),

/* === 日次ワイド化 === */
wide_day AS (
  SELECT
    k.customer_org_mapping_key,
    k.day_date,
    COALESCE(s.shipment_giv_value, 0.0)     AS shipment_giv_value,
    COALESCE(i.indirect_giv_shipments, 0.0) AS indirect_giv_shipments,
    COALESCE(g.garp_value, 0.0)             AS garp_value,
    COALESCE(o.giv_outbound, 0.0)           AS giv_outbound
  FROM all_day_keys AS k
  LEFT JOIN ship_day     AS s ON k.customer_org_mapping_key = s.customer_org_mapping_key AND k.day_date = s.day_date
  LEFT JOIN indirect_day AS i ON k.customer_org_mapping_key = i.customer_org_mapping_key AND k.day_date = i.day_date
  LEFT JOIN garp_day     AS g ON k.customer_org_mapping_key = g.customer_org_mapping_key AND k.day_date = g.day_date
  LEFT JOIN outbound_day AS o ON k.customer_org_mapping_key = o.customer_org_mapping_key AND k.day_date = o.day_date
)

/* === Final（PG Quarter集計 + FN2適用） === */
SELECT
  -- 依頼項目：FN Code / FN Name（FN2を優先し、NULLならgivの現行FNで保険）
  COALESCE(cf2.fn_code_2, gdb.fn_code_current) AS fn_code,
  COALESCE(cf2.fn_name_2, gdb.fn_name_current) AS fn_name,

  cf2.dc_ws AS dc_ws,               -- DC/WS（= jp_cust_parent_group_name）
  gdb.category AS category,
  cal.pg_quarter AS pg_quarter,
  gdb.org_code AS org_code,
  org.org2 AS org2,
  org.org3 AS org3,
  org.org4 AS org4,
  org.org5 AS org5,

  SUM(w.shipment_giv_value) AS shipment_giv_value,
  SUM(w.indirect_giv_shipments) AS indirect_giv_shipments,
  SUM(w.shipment_giv_value + w.indirect_giv_shipments) AS ships_in_giv_customer_sales,
  SUM(w.garp_value) AS garp_value,
  SUM(w.giv_outbound) AS giv_outbound

FROM wide_day AS w
JOIN cal_fy_2h AS cal
  ON w.day_date = cal.day_date
JOIN giv_dim_base AS gdb
  ON w.customer_org_mapping_key = gdb.customer_org_mapping_key
LEFT JOIN cust_fn2 AS cf2
  ON gdb.cust_key = cf2.cust_key
LEFT JOIN org
  ON gdb.org5_key = org.org5_key

GROUP BY
  COALESCE(cf2.fn_code_2, gdb.fn_code_current),
  COALESCE(cf2.fn_name_2, gdb.fn_name_current),
  cf2.dc_ws,
  gdb.category,
  cal.pg_quarter,
  gdb.org_code,
  org.org2, org.org3, org.org4, org.org5

ORDER BY
  fn_code,
  pg_quarter,
  category;

In [0]:
%sql
-- ============================================
-- 診断1：CTEごとの行数 & giv_dim_baseに存在しないキー件数（ヒントなし）
-- ============================================
WITH
/* === FY2425 2H の日付集合（DATEキーに統一） === */
cal_fy_2h AS (
  SELECT
    to_date(c.day_date) AS day_date,
    c.fy_quarter_name   AS pg_quarter
  FROM hive_metastore.japan_gold.calender_dim_vw AS c
  WHERE c.fy_year_name = 'FY2425'
    AND c.fy_half_year_name = '2H'
),

/* === Shipment：Quarter × customer_org_mapping_key === */
ship_qtr AS (
  SELECT
    sh.customer_org_mapping_key AS customer_org_mapping_key,
    cal.pg_quarter              AS pg_quarter,
    SUM(sh.gross_transact_amt)  AS shipment_giv_value
  FROM hive_metastore.japan_gold.shipments_fct_vw AS sh
  JOIN cal_fy_2h AS cal
    ON to_date(sh.day_date) = cal.day_date
  GROUP BY
    sh.customer_org_mapping_key,
    cal.pg_quarter
),

/* === Indirect Ship：Quarter × customer_org_mapping_key === */
indirect_qtr AS (
  SELECT
    isd.customer_org_mapping_key AS customer_org_mapping_key,
    cal.pg_quarter               AS pg_quarter,
    SUM(isd.gross_transact_amt)  AS indirect_giv_shipments
  FROM hive_metastore.japan_linkedexcel.indirect_ship_day_fct_vw AS isd
  JOIN cal_fy_2h AS cal
    ON to_date(isd.day_date) = cal.day_date
  GROUP BY
    isd.customer_org_mapping_key,
    cal.pg_quarter
),

/* === GARP：Quarter × customer_org_mapping_key === */
garp_qtr AS (
  SELECT
    g.customer_org_mapping_key AS customer_org_mapping_key,
    cal.pg_quarter             AS pg_quarter,
    SUM(g.garp_value)          AS garp_value
  FROM hive_metastore.japan_gold.giv_garp_dim_vw AS g
  JOIN cal_fy_2h AS cal
    ON to_date(g.day_date) = cal.day_date
  GROUP BY
    g.customer_org_mapping_key,
    cal.pg_quarter
),

/* === Outbound：Quarter × customer_org_mapping_key === */
outbound_qtr AS (
  SELECT
    go.customer_org_mapping_key AS customer_org_mapping_key,
    cal.pg_quarter              AS pg_quarter,
    SUM(go.giv_outbound)        AS giv_outbound
  FROM hive_metastore.japan_gold.giv_outbound_vw AS go
  JOIN cal_fy_2h AS cal
    ON to_date(go.day_date) = cal.day_date
  GROUP BY
    go.customer_org_mapping_key,
    cal.pg_quarter
),

/* === Quarter×mappingキーの母集合 === */
all_qtr_keys AS (
  SELECT customer_org_mapping_key, pg_quarter FROM ship_qtr
  UNION
  SELECT customer_org_mapping_key, pg_quarter FROM indirect_qtr
  UNION
  SELECT customer_org_mapping_key, pg_quarter FROM garp_qtr
  UNION
  SELECT customer_org_mapping_key, pg_quarter FROM outbound_qtr
),

/* === Quarterワイド化 === */
wide_qtr AS (
  SELECT
    k.customer_org_mapping_key                  AS customer_org_mapping_key,
    k.pg_quarter                                AS pg_quarter,
    COALESCE(s.shipment_giv_value, 0.0)         AS shipment_giv_value,
    COALESCE(i.indirect_giv_shipments, 0.0)     AS indirect_giv_shipments,
    COALESCE(g.garp_value, 0.0)                 AS garp_value,
    COALESCE(o.giv_outbound, 0.0)               AS giv_outbound
  FROM all_qtr_keys AS k
  LEFT JOIN ship_qtr     AS s
    ON k.customer_org_mapping_key = s.customer_org_mapping_key
   AND k.pg_quarter = s.pg_quarter
  LEFT JOIN indirect_qtr AS i
    ON k.customer_org_mapping_key = i.customer_org_mapping_key
   AND k.pg_quarter = i.pg_quarter
  LEFT JOIN garp_qtr     AS g
    ON k.customer_org_mapping_key = g.customer_org_mapping_key
   AND k.pg_quarter = g.pg_quarter
  LEFT JOIN outbound_qtr AS o
    ON k.customer_org_mapping_key = o.customer_org_mapping_key
   AND k.pg_quarter = o.pg_quarter
),

/* === giv_dimension_dim_vw（キー欠損診断用） === */
giv_dim_base AS (
  SELECT
    gd.customer_org_mapping_key AS customer_org_mapping_key
  FROM hive_metastore.japan_gold.giv_dimension_dim_vw AS gd
),

/* === 診断サマリ === */
diag AS (
  SELECT
    (SELECT COUNT(*) FROM cal_fy_2h)    AS cal_days,
    (SELECT COUNT(*) FROM ship_qtr)     AS ship_qtr_rows,
    (SELECT COUNT(*) FROM indirect_qtr) AS indirect_qtr_rows,
    (SELECT COUNT(*) FROM garp_qtr)     AS garp_qtr_rows,
    (SELECT COUNT(*) FROM outbound_qtr) AS outbound_qtr_rows,
    (SELECT COUNT(*) FROM all_qtr_keys) AS all_qtr_keys_rows,
    (SELECT COUNT(*) FROM wide_qtr)     AS wide_qtr_rows,
    (SELECT COUNT(*) FROM giv_dim_base) AS giv_dim_rows,

    (SELECT COUNT(*)
       FROM wide_qtr AS wq
       LEFT ANTI JOIN giv_dim_base AS gdb
         ON wq.customer_org_mapping_key = gdb.customer_org_mapping_key
    ) AS missing_in_giv_dim,

    (SELECT COUNT(*)
       FROM giv_dim_base AS gdb
       LEFT ANTI JOIN wide_qtr AS wq
         ON wq.customer_org_mapping_key = gdb.customer_org_mapping_key
    ) AS missing_in_wide_qtr
)
SELECT * FROM diag;

In [0]:
# How to Get Data from MIRAI using PySpark

# =========================================================================
# 0. 「おまじない」（ライブラリの準備）
# =========================================================================
# 複雑なデータ処理を行うための「道具箱」を読み込みます。
# ※Databricksでは省略しても動きますが、書いておくと安心です。
from pyspark.sql import SparkSession

# =========================================================================
# 1: 「どのデータを」「どこから」持ってくるか決める（SQLの設定）
# =========================================================================

# 抽出したい内容を、以下の「"""」と「"""」の間にSQLで書きます。
# ※いつものSQLツールで書いたクエリをそのままコピー＆ペーストしてOKです。
sql_query = """
SELECT 
    * -- すべてのカラム（項目）を選択
FROM 
    samples.tpch.customer       -- 【書き換え：ここに取り出し元のテーブル名を入れる】
WHERE 
    c_mktsegment = 'HOUSEHOLD'  -- 【書き換え：抽出したい条件を絞り込む】
LIMIT 100                       -- 最初はテストとして100件だけで試すのがおすすめです
"""

# =========================================================================
# 2: コンピュータにSQLを実行させる
# =========================================================================

# spark.sql() という命令を使って、上のSQLを実行し、結果を「df」という箱に入れます。
# 「df」は「データフレーム（表形式のデータ）」の略称です。
df = spark.sql(sql_query)

# 実行した結果が正しいか、最初の5行だけ画面に表形式で表示して確認します。
# これにより、保存前に意図した通りのデータかチェックできます。
display(df.limit(5))
